# AIRR Backend Parity Audit

Validate that germlines backend produces identical results to G3 backend.

In [6]:
import os
import pandas as pd
from sadie.airr import Airr

In [7]:
df = pd.read_csv("20260112_HCV_DB_example.csv")
print(f"Loaded {len(df)} sequences")
df.head()

Loaded 96 sequences


,mAb_name,epitope,epitope_subclass,neutralizing,PDB_IDs,subject_ID,species,vaccine,infection_status,Reference,VH_ortholog,D_ortholog,VL_ortholog,cdr3_aa_heavy_length,contact_residues,id,sequence_id_heavy,sequence_heavy,reference_name_heavy,locus_heavy,stop_codon_heavy,vj_in_frame_heavy,v_frameshift_heavy,productive_heavy,rev_comp_heavy,complete_vdj_heavy,v_call_top_heavy,v_call_heavy,d_call_top_heavy,d_call_heavy,j_call_top_heavy,j_call_heavy,c_call_heavy,sequence_alignment_heavy,germline_alignment_heavy,sequence_alignment_aa_heavy,germline_alignment_aa_heavy,v_alignment_start_heavy,v_alignment_end_heavy,d_alignment_start_heavy,d_alignment_end_heavy,j_alignment_start_heavy,j_alignment_end_heavy,c_alignment_start_heavy,c_alignment_end_heavy,v_sequence_alignment_heavy,v_sequence_alignment_aa_heavy,v_germline_alignment_heavy,v_germline_alignment_aa_heavy,d_sequence_alignment_heavy,d_sequence_alignment_aa_heavy,d_germline_alignment_heavy,d_germline_alignment_aa_heavy,j_sequence_alignment_heavy,j_sequence_alignment_aa_heavy,j_germline_alignment_heavy,j_germline_alignment_aa_heavy,c_sequence_alignment_heavy,c_sequence_alignment_aa_heavy,c_germline_alignment_heavy,c_germline_alignment_aa_heavy,fwr1_heavy,fwr1_aa_heavy,cdr1_heavy,cdr1_aa_heavy,fwr2_heavy,fwr2_aa_heavy,cdr2_heavy,cdr2_aa_heavy,fwr3_heavy,fwr3_aa_heavy,fwr4_heavy,fwr4_aa_heavy,cdr3_heavy,cdr3_aa_heavy,junction_heavy,junction_length_heavy,junction_aa_heavy,junction_aa_length_heavy,v_score_heavy,d_score_heavy,j_score_heavy,c_score_heavy,v_cigar_heavy,d_cigar_heavy,j_cigar_heavy,c_cigar_heavy,v_support_heavy,d_support_heavy,j_support_heavy,c_support_heavy,v_identity_heavy,d_identity_heavy,j_identity_heavy,c_identity_heavy,v_sequence_start_heavy,v_sequence_end_heavy,v_germline_start_heavy,v_germline_end_heavy,d_sequence_start_heavy,d_sequence_end_heavy,d_germline_start_heavy,d_germline_end_heavy,j_sequence_start_heavy,j_sequence_end_heavy,j_germline_start_heavy,j_germline_end_heavy,c_sequence_start_heavy,c_sequence_end_heavy,c_germline_start_heavy,c_germline_end_heavy,fwr1_start_heavy,fwr1_end_heavy,cdr1_start_heavy,cdr1_end_heavy,fwr2_start_heavy,fwr2_end_heavy,cdr2_start_heavy,cdr2_end_heavy,fwr3_start_heavy,fwr3_end_heavy,fwr4_start_heavy,fwr4_end_heavy,cdr3_start_heavy,cdr3_end_heavy,np1_heavy,np1_length_heavy,np2_heavy,np2_length_heavy,liable_heavy,vdj_nt_heavy,vdj_aa_heavy,v_mutation_heavy,v_mutation_aa_heavy,d_mutation_heavy,d_mutation_aa_heavy,j_mutation_heavy,j_mutation_aa_heavy,v_penalty_heavy,d_penalty_heavy,j_penalty_heavy,germline_alignment_aa_corrected_heavy,v_germline_alignment_aa_corrected_heavy,scheme_heavy,mutations_heavy,sequence_id_light,sequence_light,reference_name_light,locus_light,stop_codon_light,vj_in_frame_light,v_frameshift_light,productive_light,rev_comp_light,complete_vdj_light,v_call_top_light,v_call_light,d_call_top_light,d_call_light,j_call_top_light,j_call_light,c_call_light,sequence_alignment_light,germline_alignment_light,sequence_alignment_aa_light,germline_alignment_aa_light,v_alignment_start_light,v_alignment_end_light,d_alignment_start_light,d_alignment_end_light,j_alignment_start_light,j_alignment_end_light,c_alignment_start_light,c_alignment_end_light,v_sequence_alignment_light,v_sequence_alignment_aa_light,v_germline_alignment_light,v_germline_alignment_aa_light,d_sequence_alignment_light,d_sequence_alignment_aa_light,d_germline_alignment_light,d_germline_alignment_aa_light,j_sequence_alignment_light,j_sequence_alignment_aa_light,j_germline_alignment_light,j_germline_alignment_aa_light,c_sequence_alignment_light,c_sequence_alignment_aa_light,c_germline_alignment_light,c_germline_alignment_aa_light,fwr1_light,fwr1_aa_light,cdr1_light,cdr1_aa_light,fwr2_light,fwr2_aa_light,cdr2_light,cdr2_aa_light,fwr3_light,fwr3_aa_light,fwr4_light,fwr4_aa_light,cdr3_light,cdr3_aa_light,junction_light,junction_length_light,junction_aa_light,junction_aa_length_light,v_score_light,d_score_light,j_score_light,c_score_light,v_cigar_light,d_cigar_light,j

In [8]:
sequences = df[["sequence_id_heavy", "sequence_heavy"]].dropna()
sequences = sequences.rename(columns={"sequence_id_heavy": "sequence_id", "sequence_heavy": "sequence"})
print(f"Prepared {len(sequences)} sequences for annotation")
sequences.head()

Prepared 96 sequences for annotation


,sequence_id,sequence
0,0,CAGGTGCAGCTGGTGCAGTCTGGGGCTGAGGTGAAGAAGCCTGGGT...
1,48,caggtgcagctggtgcagtcaggggctgaggtgaagaagcctgggg...
2,50,caggtgcagctgcaggagtcggggccaggactgataaagtcttcac...
3,52,caggtgcagctggtgcagtctggggctgaggtgaggaagcctgggg...
4,54,caggtgcagctggtgcagtctggggctgaggtgaagaagcctgggg...


## Run with Germlines Backend

In [9]:
os.environ["SADIE_USE_GERMLINES_MODULE"] = "true"
airr_germlines = Airr("human")
result_germlines = airr_germlines.run_dataframe(sequences, seq_id_field="sequence_id", seq_field="sequence")
if "source" in result_germlines.columns:
    result_germlines = result_germlines.drop(columns=["source"])
print(f"Germlines backend: {len(result_germlines)} results, {len(result_germlines.columns)} columns")
result_germlines.head()

/Users/tmsincomb/sadie/src/sadie/airr/igblast/germline.py:221: UserWarning: C gene directory not found for human
  warnings.warn(f"C gene directory not found for {self.name}", UserWarning)
/Users/tmsincomb/sadie/src/sadie/airr/igblast/igblast.py:625: UserWarning: /Users/tmsincomb/sadie/src/sadie/germlines/igblast/Ig/internal_data/human/human_C is not found, No C gene segment
  warnings.warn(f"{path} is not found, No C gene segment", UserWarning)


ValueError: 45     0
46     2
57     6
58     8
59    10
60    12
69     0
70     2
75     8
76     4
77     6
78    10
86    11
90    19
92    10
93    14
94    16
95    12
Name: sequence_id, dtype: int64 is duplicated. Nees to be unique

## Run with G3 Backend

In [ ]:
os.environ["SADIE_USE_GERMLINES_MODULE"] = "false"
airr_g3 = Airr("human")
result_g3 = airr_g3.run_dataframe(sequences, seq_id_field="sequence_id", seq_field="sequence")
if "source" in result_g3.columns:
    result_g3 = result_g3.drop(columns=["source"])
print(f"G3 backend: {len(result_g3)} results, {len(result_g3.columns)} columns")
result_g3.head()

## Compare Results

In [ ]:
germlines_cols = set(result_germlines.columns)
g3_cols = set(result_g3.columns)

print(f"Germlines columns: {len(germlines_cols)}")
print(f"G3 columns: {len(g3_cols)}")
print(f"\nOnly in germlines: {germlines_cols - g3_cols}")
print(f"Only in G3: {g3_cols - germlines_cols}")
print(f"Common columns: {len(germlines_cols & g3_cols)}")

In [ ]:
common_cols = sorted(germlines_cols & g3_cols)
result_germlines_aligned = result_germlines[common_cols].sort_values("sequence_id").reset_index(drop=True)
result_g3_aligned = result_g3[common_cols].sort_values("sequence_id").reset_index(drop=True)

In [ ]:
differences = {}
for col in common_cols:
    mask = result_germlines_aligned[col].fillna("").astype(str) != result_g3_aligned[col].fillna("").astype(str)
    diff_count = mask.sum()
    if diff_count > 0:
        differences[col] = diff_count

if differences:
    print(f"Found differences in {len(differences)} columns:")
    for col, count in sorted(differences.items(), key=lambda x: -x[1]):
        print(f"  {col}: {count} differences")
else:
    print("No differences found - backends produce identical results")

In [ ]:
if differences:
    for col in list(differences.keys())[:5]:
        print(f"\n=== {col} ===")
        mask = result_germlines_aligned[col].fillna("").astype(str) != result_g3_aligned[col].fillna("").astype(str)
        sample_idx = mask[mask].index[:3]
        for idx in sample_idx:
            seq_id = result_germlines_aligned.loc[idx, "sequence_id"]
            print(f"  Sequence: {seq_id}")
            print(f"    Germlines: {result_germlines_aligned.loc[idx, col]}")
            print(f"    G3:        {result_g3_aligned.loc[idx, col]}")

## Summary

In [ ]:
total_values = len(common_cols) * len(result_germlines_aligned)
total_diffs = sum(differences.values()) if differences else 0
parity_pct = (1 - total_diffs / total_values) * 100 if total_values > 0 else 100

print("=" * 50)
print("AUDIT SUMMARY")
print("=" * 50)
print(f"Sequences tested: {len(sequences)}")
print(f"Common columns: {len(common_cols)}")
print(f"Total values compared: {total_values}")
print(f"Values with differences: {total_diffs}")
print(f"Parity: {parity_pct:.2f}%")
print("=" * 50)

if parity_pct == 100:
    print("PASS: Backends produce identical results")
else:
    print(f"FAIL: {len(differences)} columns have differences")